# Data Collection,Environment Setup,and Initial Cleaning

## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"C:\Users\Dell\Downloads\Superstore.csv")
df.head()



In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()


## 2.Handle 'NULL'&Duplicate Values

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df['Sales']=df['Sales'].fillna(value=df['Sales'].mean())

In [ ]:
df['Sales']

In [ ]:
df.isnull().sum()

In [ ]:
# Percentage of Missing Values
missing_percentage = (df.isnull().sum() / len(df)) * 100

In [ ]:
missing_percentage

In [ ]:
# Check Negative Sales Values
negative_sales = df[df['Sales'] < 0]

# Display Result
if negative_sales.empty:
    print("No Negative Sales Records Found")
else:
    print("Negative Sales Records:")
    print(negative_sales)

In [ ]:
# Check Invalid Quantity Values
invalid_quantity = df[df['Quantity'] <= 0]

# Display Result
if invalid_quantity.empty:
    print("No Invalid Quantity Records Found")
else:
    print("Invalid Quantity Records:")
    print(invalid_quantity)

In [ ]:
#"Data Types Before Conversion:
df.dtypes


In [ ]:
# Convert Numeric Columns if Needed
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce')

In [ ]:
#Data Types After Validation:"
df.dtypes

## OUTLIER DETECTION

In [ ]:
# Using IQR Method for Sales Column
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[(df['Sales'] < lower_limit) | 
              (df['Sales'] > upper_limit)]

outliers.shape[0]


In [ ]:
num_cols=df.select_dtypes(include=np.number).columns

for col in num_cols:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f"Outliers in {col}")
    plt.show()

## RFM Analysis

In [ ]:
df['Order Date']=pd.to_datetime(df['Order Date'])

In [ ]:
reference_date = df['Order Date'].max() + pd.Timedelta(days=1)

In [ ]:
rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (reference_date - x.max()).days,  # Recency
    'Order ID': 'nunique',                                   # Frequency
    'Sales': 'sum'                                       # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [ ]:
# Recency (lower is better → reverse labels)
rfm['R_score'] = pd.qcut(
    rfm['Recency'],
    q=4,
    labels=[4,3,2,1],
    duplicates='drop'
)

# Frequency (higher is better)
rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    q=4,
    labels=[1,2,3,4]
)

# Monetary (higher is better)
rfm['M_score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    q=4,
    labels=[1,2,3,4]
)

In [ ]:
rfm['RFM_Score'] = (rfm['R_score'].astype(str) +
                    rfm['F_score'].astype(str) +
                    rfm['M_score'].astype(str))

rfm.head()

## CUSTOMER SEGMENTATION¶

In [ ]:
def customer_segment(row):

    if row['RFM_Score'] == '444':
        return 'Best Customers'

    elif row['R_score'] == 4:
        return 'Recent Customers'

    elif row['F_score'] == 4:
        return 'Loyal Customers'

    elif row['M_score'] == 4:
        return 'High Spending Customers'

    else:
        return 'Regular Customers'


In [ ]:
# Apply Segmentation
rfm['Customer Segment'] = rfm.apply(customer_segment,
                                    axis=1)


In [ ]:
# SEGMENT COUNTS
segment_counts = rfm['Customer Segment'].value_counts()

segment_counts

In [ ]:
rfm.head()

In [ ]:
# Save Cleaned Dataset
df.to_csv("Cleaned Superstore.csv",
            index=False)


print("Files Saved Successfully")